# Label-efficient EuroSAT classification — run every experiment

Runs the whole benchmark on a **free Kaggle GPU** (T4 x2 recommended; Colab T4 also works).

**Before you start (Kaggle)**
1. *Settings → Accelerator → GPU T4 x2*, and *Settings → Internet → On* (needs a phone-verified account).
2. Create a free Hugging Face token (read access) and add it under *Add-ons → Secrets* as `HF_TOKEN`.
3. On Hugging Face, open these pages and accept the licence (instant for most accounts):
   [timm/vit_large_patch16_dinov3.sat493m](https://huggingface.co/timm/vit_large_patch16_dinov3.sat493m),
   [timm/vit_large_patch16_dinov3.lvd1689m](https://huggingface.co/timm/vit_large_patch16_dinov3.lvd1689m),
   [timm/vit_base_patch16_dinov3.lvd1689m](https://huggingface.co/timm/vit_base_patch16_dinov3.lvd1689m)
   (and [google/gemma-3-4b-it](https://huggingface.co/google/gemma-3-4b-it) if you want the optional Gemma run).

**Tip:** use *Save Version → Save & Run All (Commit)*. It runs in the background for up to 12 h and keeps
everything in `/kaggle/working`, so a closed browser tab doesn't kill the run. Set `STAGES` below to split
the work across several commits. Every script skips runs already in `results/runs.csv`, so re-running is safe.

Time estimates below are rough guesses for a T4; the real numbers end up in `results/runs.csv`.

In [ ]:
# Which parts to run. A: encoders (~1.5-2 h)  B: LoRA fine-tuning (~1-1.5 h)  C: VLMs (~2-3 h)  D: report
STAGES = "ABCD"

In [ ]:
import os, subprocess
if not os.path.exists("eurosat-foundation-models"):
    subprocess.run(["git", "clone", "https://github.com/kingmars2022/eurosat-foundation-models.git"], check=True)
os.chdir("eurosat-foundation-models")
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Hugging Face token from Kaggle or Colab secrets
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if token:
    os.environ["HF_TOKEN"] = token
    print("HF token loaded")
else:
    print("No HF_TOKEN secret found - gated models (DINOv3, Gemma) will fail to download")

In [ ]:
# Optional: continue from an earlier run. Attach the previous results as a Kaggle dataset and point to it.
PREVIOUS_RESULTS = None   # e.g. "/kaggle/input/eurosat-fm-results/results"
if PREVIOUS_RESULTS:
    !cp -rn {PREVIOUS_RESULTS}/. results/

## Data — EuroSAT RGB (27,000 Sentinel-2 tiles, 10 classes)

In [ ]:
!python -m eurofm.prepare_data --source torchvision --root data/raw

## Stage A — frozen encoders
Extract features once per encoder, then zero-shot (image-text models), linear probes at every label budget,
the from-scratch CNN reference, and export the head used by the Gradio demo.

In [ ]:
if "A" in STAGES:
    !python -m eurofm.extract_features
    !python -m eurofm.zero_shot
    !python -m eurofm.linear_probe --budgets 1,5,10,50,100,500 --seeds 0 1 2
    !python -m eurofm.linear_probe --encoders dinov3_vitl16_sat --budgets all --seeds 0 --export-head
    !python -m eurofm.finetune --mode scratch --budgets 1,5,10,50,100,500 --seeds 0 1 2

## Stage B — LoRA fine-tuning of the satellite DINOv3 ViT-L

In [ ]:
if "B" in STAGES:
    !python -m eurofm.finetune --mode lora --encoders dinov3_vitl16_sat --budgets 5,50,500 --seeds 0 1 2

## Stage C — vision-language models (Qwen3.5)
Zero-shot across model sizes, in-context 1-shot, then LoRA fine-tuning of the 2B model.
If a model prints `nan` or garbage answers in float16, re-run it with `--dtype float32` (0.8B/2B fit) or `--load-in-4bit`.

In [ ]:
if "C" in STAGES:
    !python -m eurofm.vlm eval --model qwen3.5-0.8b
    !python -m eurofm.vlm eval --model qwen3.5-2b
    !python -m eurofm.vlm eval --model qwen3.5-4b
    !python -m eurofm.vlm eval --model qwen3.5-9b --load-in-4bit --batch-size 8
    !python -m eurofm.vlm eval --model qwen3.5-4b --shots 1 --batch-size 4 --seeds 0 1 2
    !python -m eurofm.vlm finetune --model qwen3.5-2b --budgets 5,50 --seeds 0 1 2
    # optional: !python -m eurofm.vlm eval --model gemma3-4b

## Stage D — tables and figures

In [ ]:
if "D" in STAGES:
    !python -m eurofm.report
    !python app/app.py --export-examples
    from IPython.display import Markdown, Image, display
    display(Markdown(open("results/summary.md").read()))
    display(Image("results/figures/learning_curves.png"))

In [ ]:
# Bundle results for download (Kaggle: Output tab; Colab: Files pane). Commit them to the repo afterwards.
!zip -qr /kaggle/working/results.zip results app/examples 2>/dev/null || zip -qr results.zip results app/examples
print("done")